# 3D driven MHD turbulence with Ornstein-Uhlenbeck forcing.

In [ ]:
from autocvd import autocvd
autocvd(num_gpus=1)
# ruff: noqa: E402
# =======================

# general
from pathlib import Path

# jax
import jax.numpy as jnp

# plotting
import matplotlib.pyplot as plt

# astronomix constants
from astronomix import (
    FINITE_DIFFERENCE,
    PERIODIC_BOUNDARY,
)
from astronomix.option_classes.simulation_config import ISOTHERMAL

# astronomix containers
from astronomix import (
    SimulationConfig,
    SimulationParams,
    BoundarySettings,
    BoundarySettings1D,
)
from astronomix._modules._turbulent_forcing._turbulent_forcing_options import (
    TurbulentForcingConfig,
    TurbulentForcingParams,
)

# astronomix functions
from astronomix import (
    time_integration,
    get_registered_variables,
    construct_primitive_state,
    finalize_config,
    initialize_interface_fields,
)

In [ ]:
figures_dir = Path("figures")

figures_dir.mkdir(exist_ok=True)

configure the simulation — isothermal, magnetized, periodic box with OU forcing

In [ ]:
sound_speed = 0.5          # sets the sonic Mach number (v_rms ~ 1)

B_0 = 0.1                  # initial uniform field, sets the Alfvénic Mach number

num_cells = 64

config = SimulationConfig(
    solver_mode = FINITE_DIFFERENCE,
    equation_of_state = ISOTHERMAL,
    mhd = True,
    progress_bar = True,
    dimensionality = 3,
    num_cells = num_cells,
    box_size = 1.0,
    boundary_settings = BoundarySettings(
        BoundarySettings1D(PERIODIC_BOUNDARY, PERIODIC_BOUNDARY),
        BoundarySettings1D(PERIODIC_BOUNDARY, PERIODIC_BOUNDARY),
        BoundarySettings1D(PERIODIC_BOUNDARY, PERIODIC_BOUNDARY),
    ),
    turbulent_forcing_config = TurbulentForcingConfig(
        turbulent_forcing = True,
    ),
)

registered_variables = get_registered_variables(config)

run for a few turbulent crossing times (t_cross = (box_size / 2) / v_rms ~ 0.5)

In [ ]:
params = SimulationParams(
    C_cfl = 1.5,
    isothermal_sound_speed = sound_speed,
    t_end = 5.0 * 0.5,
    minimum_density = 0.02,
    turbulent_forcing_params = TurbulentForcingParams(
        energy_injection_rate = 1.65,
    ),
)

uniform medium threaded by a uniform field along z, at rest

In [ ]:
rho = jnp.ones((num_cells, num_cells, num_cells))

u_x = jnp.zeros_like(rho)

u_y = jnp.zeros_like(rho)

u_z = jnp.zeros_like(rho)

B_x = jnp.zeros_like(rho)

B_y = jnp.zeros_like(rho)

B_z = B_0 * jnp.ones_like(rho)

bxb, byb, bzb = initialize_interface_fields(B_x, B_y, B_z)

initial_state = construct_primitive_state(
    config = config,
    registered_variables = registered_variables,
    density = rho,
    velocity_x = u_x,
    velocity_y = u_y,
    velocity_z = u_z,
    magnetic_field_x = B_x,
    magnetic_field_y = B_y,
    magnetic_field_z = B_z,
    interface_magnetic_field_x = bxb,
    interface_magnetic_field_y = byb,
    interface_magnetic_field_z = bzb,
)

config = finalize_config(config, initial_state.shape)

run the simulation

In [ ]:
final_state = time_integration(initial_state, config, params, registered_variables)

plot central slices of density and velocity / magnetic-field magnitude

In [ ]:
z = num_cells // 2

velocity = final_state[jnp.array(registered_variables.velocity_index)][:, :, :, z]

magnetic_field = final_state[jnp.array(registered_variables.magnetic_index)][:, :, :, z]

density = final_state[registered_variables.density_index][:, :, z]

speed = jnp.sqrt(jnp.sum(velocity ** 2, axis=0))

b_magnitude = jnp.sqrt(jnp.sum(magnetic_field ** 2, axis=0))

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

for ax, field, title in zip(axs, (density, speed, b_magnitude), ("Density", "|v|", "|B|")):
    im = ax.imshow(field.T, origin="lower", cmap="viridis")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.savefig(figures_dir / "mhd_turbulence.png", dpi=200, bbox_inches="tight")